<a href="https://colab.research.google.com/github/Alemisa/Machine-Learning_Practical-Sessions/blob/main/Reinforcement_L_implement_a_Tic_Tac_Toe_game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

`Problem:Using Reinforcement Learning implement a Tic-Tac-Toe game. You will be playing this game against the machine learning model.`

`*Key points:`
     * Algorithm: Q-learning
     * Agent: Machine (plays O)
     * Human: You (play X)
     * Environment: Tic-Tac-Toe board
     * Training: Agent trains by playing against itself
     * After training: You play against the trained agent

In [ ]:
import random
import pickle
# -------------------------
# Tic-Tac-Toe Environment
# -------------------------
class TicTacToe:
    def __init__(self):
        self.board = [' '] * 9

    def reset(self):
        self.board = [' '] * 9
        return self.get_state()

    def get_state(self):
        return ''.join(self.board)

    def available_actions(self):
        return [i for i in range(9) if self.board[i] == ' ']

    def make_move(self, action, player):
        self.board[action] = player

    def check_winner(self):
        win_states = [
            (0,1,2),(3,4,5),(6,7,8),
            (0,3,6),(1,4,7),(2,5,8),
            (0,4,8),(2,4,6)
        ]
        for a,b,c in win_states:
            if self.board[a] == self.board[b] == self.board[c] != ' ':
                return self.board[a]
        if ' ' not in self.board:
            return 'Draw'
        return None

    def render(self):
        print()
        for i in range(0, 9, 3):
            print(self.board[i], "|", self.board[i+1], "|", self.board[i+2])
        print()
# -------------------------
# Q-Learning Agent
# -------------------------
class QAgent:
    def __init__(self, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.q = {}
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def get_q(self, state, action):
        return self.q.get((state, action), 0.0)

    def choose_action(self, state, actions):
        if random.random() < self.epsilon:
            return random.choice(actions)
        q_vals = [(self.get_q(state, a), a) for a in actions]
        return max(q_vals)[1]

    def update(self, state, action, reward, next_state, next_actions):
        max_q = max([self.get_q(next_state, a) for a in next_actions], default=0)
        current_q = self.get_q(state, action)
        self.q[(state, action)] = current_q + self.alpha * (
            reward + self.gamma * max_q - current_q
        )
# -------------------------
# Train the Agent
# -------------------------
def train_agent(episodes=50000):
    env = TicTacToe()
    agent = QAgent()

    for _ in range(episodes):
        state = env.reset()
        while True:
            actions = env.available_actions()
            action = agent.choose_action(state, actions)
            env.make_move(action, 'O')

            winner = env.check_winner()
            next_state = env.get_state()

            if winner == 'O':
                agent.update(state, action, 1, next_state, [])
                break
            elif winner == 'Draw':
                agent.update(state, action, 0, next_state, [])
                break

            # Opponent (random X)
            env.make_move(random.choice(env.available_actions()), 'X')
            winner = env.check_winner()

            if winner == 'X':
                agent.update(state, action, -1, next_state, [])
                break
            elif winner == 'Draw':
                agent.update(state, action, 0, next_state, [])
                break

            agent.update(state, action, 0, next_state, env.available_actions())
            state = next_state
    return agent
# -------------------------
# Human vs Machine
# -------------------------
def play_human_vs_agent(agent):
    env = TicTacToe()
    state = env.reset()

    print("You are X, Machine is O")
    env.render()

    while True:
        # Human move
        move = int(input("Enter your move (0-8): "))
        if move not in env.available_actions():
            print("Invalid move!")
            continue

        env.make_move(move, 'X')
        env.render()

        if env.check_winner() == 'X':
            print("🎉 You win!")
            break
        elif env.check_winner() == 'Draw':
            print("🤝 Draw!")
            break

        # Agent move
        action = agent.choose_action(env.get_state(), env.available_actions())
        env.make_move(action, 'O')
        print("Machine move:")
        env.render()

        if env.check_winner() == 'O':
            print("🤖 Machine wins!")
            break
        elif env.check_winner() == 'Draw':
            print("🤝 Draw!")
            break
# -------------------------
# Run Everything
# -------------------------
agent = train_agent()
play_human_vs_agent(agent)


You are X, Machine is O

  |   |  
  |   |  
  |   |  

Enter your move (0-8): 5

  |   |  
  |   | X
  |   |  

Machine move:

  |   | O
  |   | X
  |   |  

Enter your move (0-8): 1

  | X | O
  |   | X
  |   |  

Machine move:

  | X | O
  |   | X
  |   | O

Enter your move (0-8): 1
Invalid move!
Enter your move (0-8): 8
Invalid move!
Enter your move (0-8): 4

  | X | O
  | X | X
  |   | O

Machine move:

  | X | O
  | X | X
  | O | O

